# Pseudo-label road with the teacher

Runs the road teacher (`runs/teacher/weights/best.pt` on Drive) on the train and val
images of the pothole-only dataset and writes `labels.zip` and
`pseudo_road.csv` to `pseudo/teacher/` on Drive. Test images are never read.

1. Runtime → Change runtime type → T4 GPU.
2. Runtime → Run all (allow Drive access).
3. On the Mac: `python -m training.scripts.pseudo_label review`.

In [ ]:
TEACHER_RUN = "teacher"  # run folder under DRIVE_DIR/runs
DATASET = "merged-v1-base"  # zip name in DRIVE_DIR/data, without .zip
DRIVE_DIR = "/content/drive/MyDrive/road-pothole-seg"

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
# Get the latest code from GitHub (code reaches Colab only through git).
import os

if os.path.isdir("/content/road-pothole-seg"):
    !git -C /content/road-pothole-seg pull --ff-only
else:
    !git clone https://github.com/ngtdat9/road-pothole-seg.git /content/road-pothole-seg
%cd /content/road-pothole-seg
!git log --oneline -1
!pip install -q -e ".[train]"

In [ ]:
# Copy the dataset to the local disk: training from the Drive mount is very slow.
if not os.path.isdir(f"/content/{DATASET}"):
    !cp "{DRIVE_DIR}/data/{DATASET}.zip" /content/
    !unzip -q "/content/{DATASET}.zip" -d /content/
!wc -l /content/{DATASET}/*.txt

In [ ]:
weights = f"{DRIVE_DIR}/runs/{TEACHER_RUN}/weights/best.pt"
out = f"{DRIVE_DIR}/pseudo/{TEACHER_RUN}"
!python -m training.scripts.pseudo_label predict --weights '{weights}' \
    --data-root /content/{DATASET} --out '{out}'

In [ ]:
# Make sure labels.zip is fully written to Drive before the session ends.
drive.flush_and_unmount()

Done: tell Claude on the Mac, which fetches `labels.zip` from the synced Drive and draws the review sheets.